# CUDA MOEA Quality Evaluation

Load a saved DTLZ snapshot, generate its analytic Pareto front, compute IGD and HV, and plot the final front against theory.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "CMakeLists.txt").exists() and (candidate / "tests" / "evaluation").exists():
        repo_root = candidate
        break

evaluation_dir = repo_root / "tests" / "evaluation"
sys.path.insert(0, str(evaluation_dir))

from metrics import hypervolume, igd
from pareto_front import pareto_front_points
from snapshot_io import default_torch_device, feasible_objectives, generation_number, load_metadata, require_torch, resolve_generation
from visualization import plot_fronts

## Configuration

In [ ]:
run_dir = repo_root / "output" / "nsga3_convexdtlz2"
generation = "latest"
pf_points = 20000
torch_device = "auto"
feasibility_epsilon = 0.0
hv_method = "auto"
hv_samples = 16384
seed = 2887

## Load CUDA Result and Compute IGD and HV

In [ ]:
torch = require_torch()
device = default_torch_device() if torch_device == "auto" else torch_device
metadata = load_metadata(run_dir)
generation_dir = resolve_generation(run_dir, generation)
objectives = feasible_objectives(run_dir, generation, feasibility_epsilon).to(device)
reference_front = pareto_front_points(
    metadata["problem_name"], metadata["objective_count"],
    count=pf_points, device=device, seed=seed,
)
ideal, nadir = reference_front.amin(dim=0), reference_front.amax(dim=0)
span = (nadir - ideal).clamp_min(torch.finfo(reference_front.dtype).eps)
hv_reference = nadir + 0.1 * span
igd_score = igd(objectives, reference_front, device=device)
hv_score = hypervolume(
    objectives, hv_reference, method=hv_method, samples=hv_samples, seed=seed, device=device,
)

print(f"run_dir: {run_dir}")
print(f"algorithm: {metadata['algorithm_name']}")
print(f"problem: {metadata['problem_name']}")
print(f"generation: {generation_number(generation_dir)}")
print(f"feasible CUDA points: {objectives.shape[0]}")
print(f"theoretical PF points: {reference_front.shape[0]}")
print(f"IGD: {igd_score:.8g}")
print(f"HV: {hv_score:.8g}")

## Plot

In [ ]:
plot_fronts(
    objectives,
    reference_front,
    output_path=run_dir / "ParetoFront.png",
    title=(f"{metadata['algorithm_name']} on {metadata['problem_name']} "
           f"Gen={generation_number(generation_dir)} "
           f"IGD={igd_score:.4g} HV={hv_score:.4g}"),
)